In [ ]:
import os, sys, json, time, random, re
from datetime import datetime, timedelta, timezone
import subprocess

print("⚙️ Installing required libraries...")

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q",
     "app-store-scraper", "app_store_scraper", "google-generativeai"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
    "urllib3>=2", "requests>=2.31"
])

REQUIRED = [
    "google-play-scraper",
    "google-genai",
    "pandas",
    "tqdm",
    "beautifulsoup4",
    "python-dateutil",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])

print("✅ Dependencies installed.")

In [ ]:
import pandas as pd
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup
from dateutil import parser as dateparser
from google import genai

def get_secret(name, required=False):
    val = ""
    try:
        from google.colab import userdata
        val = userdata.get(name)
    except Exception:
        val = os.environ.get(name, "")
    if not val and required:
        import getpass
        print(f"\n🔑 Please enter your {name}:")
        val = getpass.getpass()
    return val or ""

GEMINI_API_KEY = get_secret("GEMINI_API_KEY", required=True)
TWITTER_BEARER_TOKEN = get_secret("TWITTER_BEARER_TOKEN", required=False)
SERPAPI_KEY = get_secret("SERPAPI_KEY", required=False)

if not GEMINI_API_KEY:
    raise ValueError("❌ GEMINI_API_KEY is required to run classification and synthesis.")

client = genai.Client(api_key=GEMINI_API_KEY)

LOOKBACK_DAYS = 180
CUTOFF_DATE = datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)

def within_lookback(dt):
    if dt is None:
        return True
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt >= CUTOFF_DATE

print("✅ Client configured.")

In [ ]:
test = client.interactions.create(model="gemini-3.1-flash", input="Say OK")
print(test.output_text)

In [ ]:
DISCOVERY_QUESTIONS = {
    1: "Why do users repeatedly buy from the same categories?",
    2: "What prevents users from exploring new categories?",
    3: "How do users discover products today?",
    4: "What role do habits play in shopping behavior?",
    5: "What information do users need before trying a new category?",
    6: "What frustrations emerge repeatedly?",
    7: "Which user segments are more likely to experiment?",
    8: "What unmet needs emerge consistently across discussions?",
}

CATEGORY_TAGS = [
    "repeat_purchase", "discovery", "pricing", "delivery", "quality",
    "search_ux", "recommendation", "trust", "variety", "habit",
    "promotion", "comparison", "awareness", "onboarding"
]

BARRIER_THEMES = [
    "trust_deficit", "price_sensitivity", "lack_of_awareness", "poor_search",
    "no_social_proof", "quality_uncertainty", "habit_inertia",
    "category_irrelevance", "overwhelming_choice", "past_bad_experience"
]

raw_reviews = []
print("✅ Taxonomies loaded.")

In [ ]:
# 4a. Google Play Store
print("\n🤖 Scraping Google Play Store reviews for Blinkit...")
try:
    from google_play_scraper import Sort, reviews as play_reviews

    continuation_token = None
    collected = []
    max_batches = 20
    for _ in range(max_batches):
        batch, continuation_token = play_reviews(
            'com.grofers.customerapp',
            lang='en', country='in', sort=Sort.NEWEST,
            count=200, continuation_token=continuation_token
        )
        if not batch:
            break
        collected.extend(batch)
        oldest_in_batch = min(r["at"] for r in batch if isinstance(r["at"], datetime))
        if oldest_in_batch.replace(tzinfo=timezone.utc) < CUTOFF_DATE:
            break
        if continuation_token is None:
            break

    kept = 0
    for r in collected:
        dt = r["at"] if isinstance(r["at"], datetime) else None
        if not within_lookback(dt):
            continue
        raw_reviews.append({
            "source": "play_store", "text": r["content"], "rating": r["score"],
            "date": dt.isoformat() if dt else "", "author": r["userName"],
            "url": "https://play.google.com/store/apps/details?id=com.grofers.customerapp"
        })
        kept += 1
    print(f"✅ Google Play Store: kept {kept} reviews within last {LOOKBACK_DAYS} days.")
except Exception as e:
    print(f"⚠️ Google Play Store scraping failed: {e}")

# 4b. Apple App Store
print("🍎 Scraping Apple App Store reviews for Blinkit (RSS feed)...")
APP_STORE_ID = 960335206
try:
    kept = 0
    stop = False
    for page in range(1, 11):
        if stop:
            break
        url = (f"https://itunes.apple.com/in/rss/customerreviews/"
               f"page={page}/id={APP_STORE_ID}/sortby=mostrecent/json")
        resp = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        if resp.status_code != 200:
            break
        data = resp.json()
        entries = data.get("feed", {}).get("entry", [])
        if not entries:
            break
        for e in entries:
            if "im:rating" not in e or "content" not in e:
                continue
            try:
                dt = dateparser.parse(e["updated"]["label"])
            except Exception:
                dt = None
            if dt and not within_lookback(dt):
                stop = True
                continue
            raw_reviews.append({
                "source": "app_store", "text": e["content"]["label"],
                "rating": float(e["im:rating"]["label"]),
                "date": dt.isoformat() if dt else "",
                "author": e.get("author", {}).get("name", {}).get("label", "unknown"),
                "url": f"https://apps.apple.com/in/app/id{APP_STORE_ID}"
            })
            kept += 1
        time.sleep(1)
    print(f"✅ Apple App Store: kept {kept} reviews within last {LOOKBACK_DAYS} days.")
except Exception as e:
    print(f"⚠️ Apple App Store scraping failed: {e}")

# 4d. Forums / general web (optional)
if SERPAPI_KEY:
    print("🌐 Searching forums/web for Blinkit discussions (via SerpAPI)...")
    search_queries = [
        "Blinkit review site:reddit.com",
        "Blinkit experience forum",
        "Blinkit complaint OR frustration",
        "Blinkit new category shopping experience",
    ]
    seen_urls = set()
    kept = 0
    try:
        for q in search_queries:
            resp = requests.get(
                "https://serpapi.com/search",
                params={"q": q, "engine": "google", "num": 10, "api_key": SERPAPI_KEY},
                timeout=15
            )
            if resp.status_code != 200:
                print(f"⚠️ SerpAPI query failed for '{q}': {resp.status_code}")
                continue
            results = resp.json().get("organic_results", [])
            for r in results:
                link = r.get("link")
                if not link or link in seen_urls:
                    continue
                seen_urls.add(link)
                try:
                    page = requests.get(link, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
                    soup = BeautifulSoup(page.text, "html.parser")
                    for tag in soup(["script", "style", "nav", "footer", "header"]):
                        tag.decompose()
                    text = " ".join(soup.get_text(separator=" ").split())
                    text = text[:3000]
                    if len(text) < 40:
                        continue
                    raw_reviews.append({
                        "source": f"web:{link.split('/')[2]}", "text": text,
                        "rating": None, "date": "", "author": "unknown", "url": link
                    })
                    kept += 1
                except Exception:
                    continue
                time.sleep(1)
        print(f"✅ Forums/web: kept {kept} pages (dates unknown — filter manually if needed).")
    except Exception as e:
        print(f"⚠️ Forum/web search failed: {e}")
else:
    print("⏭️ Skipping forums/general web — no SERPAPI_KEY set.")


In [ ]:
if not raw_reviews:
    print("❌ No reviews were scraped. Using fallback dummy review to prevent script failure.")
    raw_reviews.append({
        "source": "play_store",
        "text": "Blinkit is fast but they only recommend vegetables and grocery. I wanted to buy a charger but it looked duplicate and bad quality.",
        "rating": 3.0, "date": datetime.now(timezone.utc).isoformat(),
        "author": "Test User", "url": "https://play.google.com/"
    })

df_raw = pd.DataFrame(raw_reviews)

# Automatically load and merge pre-extracted Twitter/Reddit data if available
import glob
extracted_files = glob.glob("data/extracted_social_media.csv") + glob.glob("extracted_social_media.csv")
if extracted_files:
    print(f"📂 Found pre-extracted social media dataset: {extracted_files[0]}")
    df_extracted = pd.read_csv(extracted_files[0])
    df_raw = pd.concat([df_raw, df_extracted], ignore_index=True)
    print(f"➕ Merged {len(df_extracted)} pre-extracted reviews from social media.")

n_before = len(df_raw)

df_raw = df_raw.drop_duplicates(subset=["text"])
n_after_dedup = len(df_raw)

df_raw = df_raw.dropna(subset=["text"])
df_raw["word_count"] = df_raw["text"].str.split().str.len()
df_raw = df_raw[df_raw["word_count"] >= 3].drop(columns=["word_count"])
n_after_clean = len(df_raw)

print(f"\n🧹 Cleaning: {n_before} raw -> {n_after_dedup} after dedup -> {n_after_clean} after dropping sub-3-word reviews ({n_before - n_after_clean} total removed).")
print(f"\n📊 Combined, cleaned dataset: {len(df_raw)} unique items across sources:")
print(df_raw["source"].value_counts().to_string())

TOTAL_SAMPLE_TARGET = 3000

play = df_raw[df_raw["source"] == "play_store"]
non_play = df_raw[df_raw["source"] != "play_store"]

# Take non-play store first (all of them, or up to target if it exceeds target)
if len(non_play) > TOTAL_SAMPLE_TARGET:
    non_play_sampled = non_play.sample(n=TOTAL_SAMPLE_TARGET, random_state=42)
else:
    non_play_sampled = non_play

remaining_slots = TOTAL_SAMPLE_TARGET - len(non_play_sampled)

if remaining_slots > 0:
    play_sampled = play.sample(n=min(len(play), remaining_slots), random_state=42)
else:
    play_sampled = play.iloc[0:0]

df_sample = pd.concat([non_play_sampled, play_sampled]).reset_index(drop=True)
df_sample["id"] = range(1, len(df_sample) + 1)
reviews_list = df_sample.to_dict(orient="records")

print(f"\n⚡ Selected {len(reviews_list)} items for AI classification (prioritized other sources first).")
print(df_sample["source"].value_counts().to_string())


In [ ]:
print("\n🧠 Starting classification using Gemini Flash Lite...")

CLASSIFICATION_SYSTEM_PROMPT = """You are an expert product analyst specializing in quick-commerce and consumer behavior in India.
Your task is to analyze customer reviews and feedback about Blinkit and classify them along multiple dimensions.

IMPORTANT: Reviews may be in English, Hindi, Hinglish (Latin-script Hindi-English mix), or other
Indian languages. Classify based on the review's MEANING regardless of language. Do not skip,
refuse, or default to low-confidence scores purely because a review is not in English.

Respond with a JSON array where each element has this exact structure:
[
  {
    "id": <review_id>,
    "sentiment": "<positive|negative|neutral|mixed>",
    "category_tags": ["<tag1>", "<tag2>"],
    "barrier_themes": ["<theme1>"],
    "discovery_q_ids": [<q_number>],
    "confidence": <0.0-1.0>
  }
]
Return ONLY the JSON array. Do not wrap in markdown fences or add explanatory text."""

def build_classification_prompt(reviews_batch):
    category_tags_str = "\n".join(f"  - {item}" for item in CATEGORY_TAGS)
    barrier_themes_str = "\n".join(f"  - {item}" for item in BARRIER_THEMES)
    questions_str = "\n".join(f"  Q{qid}: {qtext}" for qid, qtext in DISCOVERY_QUESTIONS.items())

    review_entries = []
    for r in reviews_batch:
        rating_str = f" | Rating: {r.get('rating')}" if r.get("rating") else ""
        text_excerpt = str(r["text"])[:1500]
        review_entries.append(f'[ID:{r["id"]}] (Source: {r["source"]}{rating_str})\n"{text_excerpt}"')

    reviews_block = "\n\n".join(review_entries)

    return f"""Analyze the following customer reviews/feedback about Blinkit and classify each one.

CATEGORY TAGS (assign 1-4):
{category_tags_str}

BARRIER THEMES (assign 0-3 only if they discuss obstacles exploring new categories):
{barrier_themes_str}

DISCOVERY QUESTIONS (assign 1-3 QIDs that this review provides evidence for):
{questions_str}

Reviews to classify:
{reviews_block}"""

def call_gemini_with_retry(model_name, system_prompt, prompt, max_retries=6, delay=10.0):
    """Calls Gemini via the Interactions API, with exponential backoff on
    rate limits. Optimized for 15 RPM, 250k TPM, and 250 RPD."""
    for attempt in range(max_retries):
        try:
            time.sleep(delay)
            interaction = client.interactions.create(
                model=model_name,
                system_instruction=system_prompt,
                input=prompt
            )
            if interaction and interaction.output_text:
                return interaction.output_text
        except Exception as e:
            err_str = str(e)
            if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str or "Quota" in err_str:
                wait_time = delay * (2 ** attempt) + random.uniform(0.5, 1.5)
                print(f"⚠️ Rate limited. Waiting {wait_time:.1f}s (Attempt {attempt+1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                print(f"⚠️ API call error: {e}")
                time.sleep(2)
    return None

def extract_json(response_text):
    clean_text = response_text.strip()
    if clean_text.startswith("```"):
        clean_text = re.sub(r"^```(?:json)?\s*\n?", "", clean_text)
        clean_text = re.sub(r"\n?```\s*$", "", clean_text)
        clean_text = clean_text.strip()
    return clean_text

def classify_batch(batch):
    """Runs one batch through the classifier. Factored out so validation
    can reuse it for the consistency re-check."""
    prompt = build_classification_prompt(batch)
    response_text = call_gemini_with_retry('gemini-2.0-flash-lite', CLASSIFICATION_SYSTEM_PROMPT, prompt)
    if not response_text:
        return []
    clean_text = extract_json(response_text)
    try:
        parsed = json.loads(clean_text)
        return parsed if isinstance(parsed, list) else []
    except Exception as e:
        print(f"⚠️ Failed to parse JSON: {e}")
        return []

classification_checkpoint_path = "classification_checkpoint.json"
classifications_results = []

if os.path.exists(classification_checkpoint_path):
    try:
        with open(classification_checkpoint_path, "r", encoding="utf-8") as f:
            classifications_results = json.load(f)
        print(f"ℹ️ Loaded {len(classifications_results)} existing classifications from checkpoint.")
    except Exception as e:
        print(f"⚠️ Failed to load checkpoint: {e}. Starting fresh.")

already_classified_ids = {c["id"] for c in classifications_results if isinstance(c, dict) and "id" in c}

to_classify = [r for r in reviews_list if r["id"] not in already_classified_ids]
print(f"⚡ {len(to_classify)} of {len(reviews_list)} reviews need classification.")

if to_classify:
    # Batch size set to 100 to reduce total request count (under daily RPD limits)
    batch_size = 100
    for i in tqdm(range(0, len(to_classify), batch_size), desc="Classifying batches"):
        batch = to_classify[i:i+batch_size]
        result = classify_batch(batch)
        if result:
            classifications_results.extend(result)
            # Save checkpoint immediately
            with open(classification_checkpoint_path, "w", encoding="utf-8") as f:
                json.dump(classifications_results, f, indent=2)
        else:
            print(f"❌ Batch starting with ID {batch[0]['id']} failed entirely.")
            print("⚠️ A batch failed entirely. Breaking loop to preserve progress.")
            break

class_map = {c["id"]: c for c in classifications_results if isinstance(c, dict) and "id" in c}

enriched_reviews = []
for r in reviews_list:
    c = class_map.get(r["id"], {})
    enriched_reviews.append({
        "id": r["id"], "source": r["source"], "text": r["text"], "rating": r["rating"],
        "date": r["date"], "author": r["author"], "url": r["url"],
        "sentiment": c.get("sentiment", "neutral"),
        "category_tags": json.dumps(c.get("category_tags", [])),
        "barrier_themes": json.dumps(c.get("barrier_themes", [])),
        "discovery_q_ids": json.dumps(c.get("discovery_q_ids", [])),
        "confidence": c.get("confidence", 0.5)
    })

df_classified = pd.DataFrame(enriched_reviews)
df_classified.to_csv("classified_reviews.csv", index=False)
print("✅ Classification finished! Saved to 'classified_reviews.csv'.")


In [ ]:
print("\n🔍 Running validation checks...")

VALIDATION_SAMPLE_SIZE = 40
audit_sample = df_classified.sample(n=min(VALIDATION_SAMPLE_SIZE, len(df_classified)), random_state=7).copy()
audit_sample["human_agrees_sentiment"] = ""   # fill in manually: yes/no
audit_sample["human_agrees_tags"] = ""        # fill in manually: yes/no
audit_sample["notes"] = ""
audit_sample.to_csv("validation_audit_sample.csv", index=False)
print(f"📝 Exported {len(audit_sample)} rows to 'validation_audit_sample.csv' — "
      f"read the raw text, fill in agreement columns, and report the agreement % in your deck.")

CONSISTENCY_CHECK_SIZE = 20
consistency_subset = df_sample.sample(n=min(CONSISTENCY_CHECK_SIZE, len(df_sample)), random_state=99).to_dict(orient="records")
print(f"🔁 Re-classifying {len(consistency_subset)} reviews for a consistency check...")
rerun_results = classify_batch(consistency_subset)
rerun_map = {c["id"]: c for c in rerun_results if isinstance(c, dict) and "id" in c}

matches, total = 0, 0
for r in consistency_subset:
    original = class_map.get(r["id"])
    rerun = rerun_map.get(r["id"])
    if original and rerun:
        total += 1
        if original.get("sentiment") == rerun.get("sentiment"):
            matches += 1

if total > 0:
    consistency_pct = 100 * matches / total
    print(f"✅ Sentiment consistency across re-run: {matches}/{total} ({consistency_pct:.1f}%)")
else:
    print("⚠️ Consistency check inconclusive — could not match enough IDs across runs.")


In [ ]:
print("\n🧪 Synthesizing insights for the 8 Core Questions using Gemini Flash Lite...")

SYNTHESIS_SYSTEM_PROMPT = """You are a senior product strategist at Blinkit. Your task is to synthesize customer reviews into actionable insights.
Respond with a JSON object with this exact structure:
{
  "summary": "<thematic summary paragraph>",
  "key_themes": [
    {
      "title": "<theme title>",
      "description": "<1-sentence description>",
      "frequency": <count>
    }
  ],
  "representative_quotes": [
    {
      "text": "<verbatim quote>",
      "review_id": <id>,
      "source": "<source>"
    }
  ],
  "confidence": <0.0-1.0>,
  "limitations": "<any limitations/caveats, 1-2 sentences>"
}
Return ONLY the JSON object. Do not wrap in markdown fences or add explanatory text."""

synthesis_checkpoint_path = "synthesis_checkpoint.json"
synthesized_insights = []

if os.path.exists(synthesis_checkpoint_path):
    try:
        with open(synthesis_checkpoint_path, "r", encoding="utf-8") as f:
            synthesized_insights = json.load(f)
        print(f"ℹ️ Loaded {len(synthesized_insights)} existing insights from checkpoint.")
    except Exception as e:
        print(f"⚠️ Failed to load synthesis checkpoint: {e}. Starting fresh.")

already_synthesized_qids = {s["question_id"] for s in synthesized_insights if isinstance(s, dict) and "question_id" in s}

for q_id, q_text in tqdm(DISCOVERY_QUESTIONS.items(), desc="Synthesizing Questions"):
    if q_id in already_synthesized_qids:
        continue
        
    relevant_reviews = []
    for r in enriched_reviews:
        try:
            q_ids = json.loads(r["discovery_q_ids"])
            if q_id in q_ids:
                relevant_reviews.append(r)
        except Exception:
            continue

    if not relevant_reviews:
        fallback_pool = enriched_reviews
        sample_size = min(15, len(fallback_pool))
        relevant_reviews = random.sample(fallback_pool, sample_size) if sample_size > 0 else []
        print(f"  ℹ️ Q{q_id} had no directly-tagged reviews — using a random sample of {len(relevant_reviews)} as fallback context.")

    review_entries = []
    for r in relevant_reviews[:30]:
        rating_str = f" ★{r['rating']}" if r['rating'] else ""
        text_excerpt = str(r["text"])[:1500]
        review_entries.append(f'[ID:{r["id"]}] ({r["source"]}{rating_str} | {r["sentiment"]})\n"{text_excerpt}"')
    reviews_block = "\n\n".join(review_entries)

    prompt = f"""Synthesize classified customer reviews into a comprehensive insight for this discovery question.

QUESTION Q{q_id}: "{q_text}"

RELEVANT REVIEWS:
{reviews_block}"""

    # delay set to 10.0 seconds to keep tokens/min and requests/min extremely safe
    response_text = call_gemini_with_retry('gemini-3.1-flash', SYNTHESIS_SYSTEM_PROMPT, prompt, delay=10.0)
    if response_text:
        clean_text = extract_json(response_text)
        try:
            parsed = json.loads(clean_text)
            parsed["question_id"] = q_id
            parsed["question_text"] = q_text
            synthesized_insights.append(parsed)
            # Save checkpoint immediately
            with open(synthesis_checkpoint_path, "w", encoding="utf-8") as f:
                json.dump(synthesized_insights, f, indent=2)
        except Exception as e:
            print(f"⚠️ Failed to parse synthesis JSON for Q{q_id}: {e}")
    else:
        print(f"❌ Synthesis for Q{q_id} failed.")
        print("⚠️ Synthesis failed. Breaking loop to preserve progress.")
        break

df_insights = pd.DataFrame(synthesized_insights)
df_insights.to_csv("synthesized_insights.csv", index=False)
print("✅ Synthesis finished! Saved to 'synthesized_insights.csv'.")

print("\n🎉 Pipeline execution completed successfully!")
print("📁 Download from Colab file panel: 'classified_reviews.csv', 'validation_audit_sample.csv', 'synthesized_insights.csv'")
